# Limpieza de bases_samur.csv

Catálogo de las bases del SAMUR: id, nombre, distrito, coordenadas, unidades asignadas, año de apertura y tipo de base.

El CSV trae 15 filas, una por base. Al final guardamos `bases_samur_limpio.csv`, también con 15 filas: aquí no hay filas que quitar, solo valores que poner en el mismo formato.

El CSV original no se toca: el resultado se guarda aparte en `bases_samur_limpio.csv`.

In [1]:
import pandas as pd
import numpy as np
import unicodedata
from pathlib import Path

# El CSV original debe estar junto al notebook o en la carpeta data del repo.
ruta = Path('bases_samur.csv')
if not ruta.exists():
    ruta = Path('../data/bases_samur.csv')
bases = pd.read_csv(ruta)
bases.head()

,base_id,nombre,distrito,latitud,longitud,unidades_asignadas,año_apertura,tipo
0,1,Base Centro,Centro,40.4168,-3.7038,6,1985.0,principal
1,2,Base Latina,La Latina,40.4017,-3.7495,5,1990.0,secundaria
2,3,Base Carabanchel,CARABANCHEL,40.3847,-3.7352,5,1992.0,SECUNDARIA
3,4,Base Vallecas,PUENTE DE VALLECAS,40.3981,-3.6589,4,NaN,PRINCIPAL
4,5,Base Ciudad Lineal,CIUDAD LINEAL,40.4382,-3.6495,4,1998.0,satelite


## Revisión inicial

Miramos los datos antes de tocar nada: cuántas filas hay, dónde están los huecos y qué valores raros aparecen.

In [2]:
print(bases.shape)            # 15 filas, una por base
print()
print(bases.isna().sum())     # huecos en año_apertura
print()
print(f"base_id repetidos: {bases['base_id'].duplicated().sum()}")
print(f"Filas duplicadas enteras: {bases.duplicated().sum()}")
print()
print(f"Latitud min/max: {bases['latitud'].min()} / {bases['latitud'].max()}")
print(f"Longitud min/max: {bases['longitud'].min()} / {bases['longitud'].max()}")
print()
print("Tipos de base escritos de distintas formas:")
print(sorted(bases['tipo'].unique()))

(15, 8)

base_id               0
nombre                0
distrito              0
latitud               0
longitud              0
unidades_asignadas    0
año_apertura          2
tipo                  0
dtype: int64

base_id repetidos: 0
Filas duplicadas enteras: 0

Latitud min/max: -95.2 / 91.5
Longitud min/max: -3.7495 / 185.0

Tipos de base escritos de distintas formas:
['PRINCIPAL', 'SATELITE', 'SECUNDARIA', 'Secundaria', 'auxiliar', 'principal', 'satelite', 'secundaria']


## Distrito de cada base

Los distritos vienen con mayúsculas y minúsculas mezcladas (`Centro`, `CARABANCHEL`) y uno lleva artículo (`La Latina`). Usamos la misma regla que en las otras limpiezas: mayúsculas, sin tildes, sin espacios sobrantes y sin el artículo del principio; si el texto es el principio de un único nombre oficial, se completa; y si no se reconoce, se deja vacío y se lista. Nada de diccionarios cerrados, como nos pidió Bluetab.

In [3]:
OFICIALES = ["CENTRO", "ARGANZUELA", "RETIRO", "SALAMANCA", "CHAMARTIN", "TETUAN", "CHAMBERI",
             "FUENCARRAL-EL PARDO", "MONCLOA-ARAVACA", "LATINA", "CARABANCHEL", "USERA",
             "PUENTE DE VALLECAS", "MORATALAZ", "CIUDAD LINEAL", "HORTALEZA", "VILLAVERDE",
             "VILLA DE VALLECAS", "VICALVARO", "SAN BLAS-CANILLEJAS", "BARAJAS"]

def normalizar_texto(s):
    if pd.isna(s):
        return None
    s = unicodedata.normalize('NFD', str(s).strip().upper())
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    return ' '.join(s.split())

def normalizar_distrito(nombre):
    t = normalizar_texto(nombre)
    if t is None:
        return None
    for articulo in ('LA ', 'EL ', 'LOS ', 'LAS '):
        if t.startswith(articulo):
            t = t[len(articulo):]
    if t in OFICIALES:
        return t
    candidatos = [o for o in OFICIALES if o.startswith(t)]
    if len(candidatos) == 1:
        return candidatos[0]
    return None

bases['distrito'] = bases['distrito'].map(normalizar_distrito)
sin_asignar = bases[bases['distrito'].isna()]
print(f"Bases sin distrito reconocido: {len(sin_asignar)}")
if len(sin_asignar):
    print(sin_asignar)
print()
print(sorted(bases['distrito'].unique()))

Bases sin distrito reconocido: 0

['ARGANZUELA', 'BARAJAS', 'CARABANCHEL', 'CENTRO', 'CHAMARTIN', 'CIUDAD LINEAL', 'FUENCARRAL-EL PARDO', 'HORTALEZA', 'LATINA', 'MONCLOA-ARAVACA', 'PUENTE DE VALLECAS', 'TETUAN', 'USERA', 'VICALVARO', 'VILLAVERDE']


Las 15 bases quedan con un distrito oficial. `La Latina` pasa a `LATINA` quitando el artículo, y el resto solo cambia de formato (mayúsculas, sin tildes).

## Tipo de base

El tipo viene escrito con mayúsculas y minúsculas mezcladas (`principal`, `PRINCIPAL`, `Secundaria`). Solo cambia la escritura, no el significado, así que se normaliza con la misma regla de texto y ya quedan las 4 categorías reales. No hace falta ninguna equivalencia: no hay valores distintos que signifiquen lo mismo.

In [4]:
bases['tipo'] = bases['tipo'].map(normalizar_texto)
print(sorted(bases['tipo'].unique()))
print()
print(bases['tipo'].value_counts())

['AUXILIAR', 'PRINCIPAL', 'SATELITE', 'SECUNDARIA']

tipo
SECUNDARIA    9
PRINCIPAL     2
SATELITE      2
AUXILIAR      2
Name: count, dtype: int64


## Coordenadas imposibles

Tres bases tienen coordenadas imposibles:

- Base Hortaleza: latitud -95,2 (la latitud solo va de -90 a 90, y Madrid está sobre 40,4).
- Base Barajas: latitud 91,5 (también fuera de rango).
- Base Moncloa: longitud 185 (la longitud solo va de -180 a 180, y Madrid está sobre -3,7).

Comprobamos el rango de Madrid (latitud entre 40,2 y 40,6, longitud entre -3,95 y -3,5, que es donde cae el resto de bases) y lo que no entra pasa a vacío: no sabemos las coordenadas reales y no las inventamos. La base se conserva con el resto de sus datos.

In [5]:
malas = (~bases['latitud'].between(40.2, 40.6)) | (~bases['longitud'].between(-3.95, -3.5))
print("Coordenadas fuera del rango de Madrid:")
print(bases.loc[malas, ['base_id', 'nombre', 'latitud', 'longitud']].to_string(index=False))

bases.loc[malas, ['latitud', 'longitud']] = np.nan
print()
print(f"Coordenadas pasadas a vacío: {malas.sum()} bases")
print(f"Latitud min/max tras limpiar: {bases['latitud'].min()} / {bases['latitud'].max()}")
print(f"Longitud min/max tras limpiar: {bases['longitud'].min()} / {bases['longitud'].max()}")

Coordenadas fuera del rango de Madrid:
 base_id         nombre  latitud  longitud
       7 Base Hortaleza -95.2000   -3.6428
       8   Base Moncloa  40.4353  185.0000
      14   Base Barajas  91.5000   -3.5808

Coordenadas pasadas a vacío: 3 bases
Latitud min/max tras limpiar: 40.3467 / 40.4814
Longitud min/max tras limpiar: -3.7495 / -3.6089


## Año de apertura

A dos bases (Base Vallecas y Base Vicálvaro) les falta el año de apertura. No hay otra tabla del proyecto donde mirarlo, así que se quedan vacías: no inventamos el dato. El año venía como decimal (`1985.0`) por los huecos; se deja como entero.

In [6]:
bases['año_apertura'] = bases['año_apertura'].astype('Int64')  # entero que admite vacíos
print(f"Bases sin año de apertura: {bases['año_apertura'].isna().sum()}")
print(bases.loc[bases['año_apertura'].isna(), ['base_id', 'nombre']].to_string(index=False))
print()
print(f"Rango de años: {bases['año_apertura'].min()} - {bases['año_apertura'].max()}")

Bases sin año de apertura: 2
 base_id         nombre
       4  Base Vallecas
      15 Base Vicálvaro

Rango de años: 1985 - 2010


## Estadísticos y guardado

Las 15 filas quedan ordenadas por `base_id`. Miramos los estadísticos finales y guardamos.

Ojo al cruzar con `unidades_samur.csv`: `unidades_asignadas` no siempre coincide con las unidades que el catálogo de unidades tiene en ese distrito. Como en `distritos.csv` con `num_hospitales` y `base_samur`, no corregimos ninguna de las dos fuentes: solo se tiene en cuenta en `cruces_datasets.ipynb`.

In [7]:
bases = bases.sort_values('base_id').reset_index(drop=True)
# Mismo orden de columnas que el original
bases = bases[['base_id', 'nombre', 'distrito', 'latitud', 'longitud',
               'unidades_asignadas', 'año_apertura', 'tipo']]
bases.to_csv('../data_clean/bases_samur_limpio.csv', index=False)
print(bases.shape)   # (15, 8)
print()
print("Huecos que quedan (los valores que no conocemos):")
print(bases.isna().sum()[bases.isna().sum() > 0])
print()
bases.describe().round(2)

(15, 8)

Huecos que quedan (los valores que no conocemos):
latitud         3
longitud        3
año_apertura    2
dtype: int64



,base_id,latitud,longitud,unidades_asignadas,año_apertura
count,15.00,12.00,12.00,15.00,13.0
mean,8.00,40.41,-3.69,3.47,1996.77
std,4.47,0.04,0.04,1.25,7.85
min,1.00,40.35,-3.75,2.00,1985.0
25%,4.50,40.39,-3.71,2.50,1991.0
50%,8.00,40.41,-3.70,3.00,1996.0
75%,11.50,40.44,-3.67,4.00,2002.0
max,15.00,40.48,-3.61,6.00,2010.0
